# GEE Fundamentals & Setup


**Objectives**
- Confirm your local/Colab environment is ready (packages, auth)
- Authenticate and initialize Google Earth Engine (GEE) via the Python API
- Understand the four core GEE object types: `Image`, `ImageCollection`, `Geometry`/`Feature`, `FeatureCollection`
- Filter, inspect, and visualize data with `geemap`
- Run your first server-side reducer



## 1. Environment check
Run the cell below. Every import should succeed and print a version. If anything fails, install it before moving on.

In [12]:
import sys, importlib

REQUIRED = ["ee", "geemap", "numpy", "pandas", "rasterio", "geopandas", "matplotlib", "shapely"]

print(f"Python: {sys.version.split()[0]}\n")
missing = []
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
        print(f"[OK]   {pkg:<12} {ver}")
    except ImportError:
        missing.append(pkg)
        print(f"[MISS] {pkg}")

if missing:
    print(f"\nInstall missing packages, e.g.:\n  pip install {' '.join(missing)}")
else:
    print("\nEnvironment ready.")

Python: 3.13.5

[OK]   ee           1.7.18
[OK]   geemap       0.38.3
[OK]   numpy        2.4.0
[OK]   pandas       2.3.3
[OK]   rasterio     1.4.4
[OK]   geopandas    1.1.3
[OK]   matplotlib   3.10.8
[OK]   shapely      2.1.2

Environment ready.


## 2. Authenticate & initialize Earth Engine
Replace `YOUR-GEE-PROJECT-ID` with the Cloud Project you created in the pre-work slides. `ee.Authenticate()` only needs to run once per machine — it opens a browser/OAuth flow and caches a token.

In [13]:
import ee

PROJECT_ID = "lunar-linker-509416-t1"  # <-- set this

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine initialized without re-authenticating.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Authenticated and initialized.")

# Sanity check: a trivial server-side computation
print(ee.Number(1).add(1).getInfo())  # should print 2

EEException: Caller does not have required permission to use project lunar-linker-509416-t1. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=lunar-linker-509416-t1 and then retry. Propagation of the new permission may take a few minutes.

## 3. Core object #1 — `ee.Image`
An `Image` is a single raster: a stack of bands, each with its own name, data type, resolution, and projection. Nothing is downloaded — you're building a *server-side* description of a computation that only runs when you ask for results (`.getInfo()`, `.reduceRegion()`, export, or map tiles).

In [ ]:
# A single Sentinel-2 scene
img = ee.Image("COPERNICUS/S2_SR_HARMONIZED/20230115T075201_20230115T075739_T36MZE")

info = img.getInfo()
print("Band names:", img.bandNames().getInfo())
print("Projection (B4):", img.select("B4").projection().getInfo()['crs'])
print("Nominal scale (B4, m):", img.select("B4").projection().nominalScale().getInfo())
print("Acquisition date:", ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo())

## 4. Core object #2 — `ee.ImageCollection`
A stack of `Image`s sharing a common source (e.g. every Sentinel-2 scene ever captured). You almost never work with a whole collection — you **filter** it down by date, location, and metadata first, exactly like a database query.

In [ ]:
# Define a small AOI: Lake Naivasha, Kenya (used throughout this workshop)
naivasha_aoi = ee.Geometry.Rectangle([36.20, -0.83, 36.45, -0.65])

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(naivasha_aoi)
      .filterDate("2023-01-01", "2023-03-01")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))

print("Scenes matching filter:", s2.size().getInfo())

# Inspect the first scene's metadata
first = s2.first()
print("First scene ID:", first.get("system:index").getInfo())
print("Cloud %:", first.get("CLOUDY_PIXEL_PERCENTAGE").getInfo())

## 5. Core object #3 — `Geometry` and `Feature`
- `Geometry` — pure shape (Point, LineString, Polygon, Rectangle…), no attributes.
- `Feature` — a `Geometry` + a dictionary of properties.
- `FeatureCollection` — a table of `Feature`s (this is what a shapefile/GeoJSON becomes in GEE).

In [ ]:
point = ee.Geometry.Point([36.36, -0.72])              # a spot on Lake Naivasha
polygon = naivasha_aoi                                    # reuse the AOI above

feature = ee.Feature(polygon, {"name": "Naivasha AOI", "purpose": "workshop"})
fc = ee.FeatureCollection([feature])

print("Feature properties:", feature.getInfo()["properties"])
print("Area (km²):", polygon.area().divide(1e6).getInfo())

## 6. Visualization with `geemap`
`geemap` wraps `ipyleaflet`/`folium` to give you an interactive map inside the notebook, with a thin bridge to GEE's tile server.

In [ ]:
import geemap

Map = geemap.Map(center=[-0.74, 36.33], zoom=11)

s2_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
composite = s2.median().clip(naivasha_aoi)

Map.addLayer(composite, s2_vis, "S2 median composite — true color")
Map.addLayer(naivasha_aoi, {"color": "yellow"}, "Workshop AOI", opacity=0.3)
Map

## 7. Reducers — pulling numbers out of an `Image`
A reducer collapses pixels to a statistic. `reduceRegion` computes it over a geometry; `reduceRegions` does it per-feature over a `FeatureCollection`.

In [ ]:
stats = composite.select(["B4", "B3", "B2"]).reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=naivasha_aoi,
    scale=30,       # coarser scale for a quick, cheap check
    maxPixels=1e9
)
print(stats.getInfo())

## Checklist before Notebook 2
- [ ] `ee.Initialize()` runs without error and prints `2`
- [ ] You can filter an `ImageCollection` and get a non-zero `.size()`
- [ ] The interactive map renders a composite over Lake Naivasha
- [ ] `reduceRegion` returns mean/stdDev values (not `null`)

If all four are checked, you're ready for **Notebook 2 — Acquiring S1, S2, Landsat**.